# Recursive Forecasting: Correcting Feature Leakage

Jan Rathfelder pointed out an issue with my evaluation in `03_xgboost_model.ipynb`: lag and rolling features were computed on the full dataset before splitting into train/test. That means later days in the 28-day test window could pull real sales values from earlier in that same window, values that wouldn't actually be known yet in a real forecasting scenario.

`lag_28` turns out to be unaffected, since the forecast horizon (28 days) exactly matches the lag window, every `lag_28` value in the test period reaches back into real training data only. `lag_7` and `lag_14` are affected past day 8 and day 15 of the test window, respectively.

This notebook re-evaluates the model using recursive forecasting: predicting one day at a time and feeding each prediction back in as if it were real, so later predictions never see actual future values, only their own prior forecasts.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

FORECAST_HORIZON = 28

In [3]:
daily_sales = pd.read_csv('../data/processed/daily_sales_CA1_FOODS.csv')
daily_sales['date'] = pd.to_datetime(daily_sales['date'])
daily_sales = daily_sales.sort_values('date').reset_index(drop=True)

train = daily_sales[:-FORECAST_HORIZON].copy()
test = daily_sales[-FORECAST_HORIZON:].copy()

print(f"Train: {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Test:  {test['date'].min().date()} to {test['date'].max().date()}")

Train: 2011-01-29 to 2016-04-24
Test:  2016-04-25 to 2016-05-22


In [4]:
def create_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['quarter'] = df['date'].dt.quarter
    df['lag_7'] = df['total_sales'].shift(7)
    df['lag_14'] = df['total_sales'].shift(14)
    df['lag_28'] = df['total_sales'].shift(28)
    df['rolling_mean_7'] = df['total_sales'].shift(1).rolling(7).mean()
    df['rolling_mean_28'] = df['total_sales'].shift(1).rolling(28).mean()
    df['rolling_std_7'] = df['total_sales'].shift(1).rolling(7).std()
    return df

feature_cols = [
    'day_of_week', 'day_of_month', 'month', 'year',
    'week_of_year', 'is_weekend', 'quarter',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7'
]

## Retraining on training data only

Same tuned hyperparameters as `03_xgboost_model.ipynb`, but trained strictly on `train`, no test-period data involved anywhere in this fit.

In [5]:
full_data_train_only = create_features(train.copy())
full_data_train_only = full_data_train_only.dropna()

X_train = full_data_train_only[feature_cols]
y_train = full_data_train_only['total_sales']

model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)

print("Model trained on training data only.")

Model trained on training data only.


## Recursive forecasting function

Predicts one day, appends that prediction to history as if it were real, then moves to the next day. This means any lag or rolling feature that reaches into the forecast window uses a prior prediction, never a real future value.

In [6]:
def recursive_forecast(model, history_df, horizon, feature_cols):
    history = history_df.copy()
    predictions = []
    last_date = history['date'].max()

    for step in range(horizon):
        next_date = last_date + pd.Timedelta(days=step + 1)

        temp_row = pd.DataFrame({'date': [next_date], 'total_sales': [np.nan]})
        temp_history = pd.concat([history, temp_row], ignore_index=True)

        temp_features = create_features(temp_history)
        X_next = temp_features[feature_cols].iloc[[-1]]

        pred = model.predict(X_next)[0]
        predictions.append(pred)

        history = pd.concat([
            history,
            pd.DataFrame({'date': [next_date], 'total_sales': [pred]})
        ], ignore_index=True)

    return predictions

In [7]:
def mape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    mask = actual > 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

recursive_predictions = recursive_forecast(
    model=model,
    history_df=train[['date', 'total_sales']],
    horizon=FORECAST_HORIZON,
    feature_cols=feature_cols
)

recursive_mape = mape(test['total_sales'].values, np.array(recursive_predictions))
original_mape = 5.7  # single-shot MAPE from 03_xgboost_model.ipynb

print(f"Recursive (honest) MAPE: {recursive_mape:.1f}%")
print(f"Original (leaked) MAPE:  {original_mape}%")

Recursive (honest) MAPE: 5.5%
Original (leaked) MAPE:  5.7%


## Comparing against mlforecast (Nixtla)

Jan also suggested `mlforecast`, a library that handles this recursive logic internally instead of a hand-written loop. Rebuilding the same setup here as a cross-check.

In [8]:
import sys
print(sys.executable)

!{sys.executable} -m pip install mlforecast

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd
print("mlforecast imported successfully")

/usr/local/bin/python3.11

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
mlforecast imported successfully


In [9]:
mlf_train = train[['date', 'total_sales']].copy()
mlf_train['unique_id'] = 'CA1_FOODS'
mlf_train = mlf_train.rename(columns={'date': 'ds', 'total_sales': 'y'})
mlf_train = mlf_train[['unique_id', 'ds', 'y']]

print(mlf_train.head())

   unique_id         ds     y
0  CA1_FOODS 2011-01-29  3239
1  CA1_FOODS 2011-01-30  3137
2  CA1_FOODS 2011-01-31  2008
3  CA1_FOODS 2011-02-01  2258
4  CA1_FOODS 2011-02-02  2032


`mlforecast`'s built-in `date_features` only accepts simple attributes like `dayofweek` or `month`, it doesn't generate `is_weekend` or `week_of_year` on its own (and `.dt.week` isn't available in newer pandas versions anyway). Passing these in as functions instead, so the feature set actually matches what the manual model saw.

In [10]:
def is_weekend(dates):
    return (dates.dayofweek >= 5).astype(int)

def week_of_year(dates):
    return dates.isocalendar().week.astype(int)

fcst = MLForecast(
    models=[GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        random_state=42
    )],
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingMean(window_size=28), RollingStd(window_size=7)]
    },
    date_features=['dayofweek', 'day', 'month', 'year', 'quarter', is_weekend, week_of_year]
)

fcst.fit(mlf_train)
mlf_predictions_df = fcst.predict(FORECAST_HORIZON)
print(mlf_predictions_df.head())

   unique_id         ds  GradientBoostingRegressor
0  CA1_FOODS 2016-04-25                2695.331778
1  CA1_FOODS 2016-04-26                2428.786738
2  CA1_FOODS 2016-04-27                2323.594967
3  CA1_FOODS 2016-04-28                2418.171458
4  CA1_FOODS 2016-04-29                2794.605202


In [11]:
mlf_predictions = mlf_predictions_df['GradientBoostingRegressor'].values
mlf_mape = mape(test['total_sales'].values, mlf_predictions)

print(f"mlforecast recursive MAPE: {mlf_mape:.1f}%")
print(f"Manual recursive MAPE:     {recursive_mape:.1f}%")
print(f"Original (leaked) MAPE:    {original_mape}%")

mlforecast recursive MAPE: 6.0%
Manual recursive MAPE:     5.5%
Original (leaked) MAPE:    5.7%


## Summary

| Approach | MAPE |
|---|---|
| Original (single-shot, leaked) | 5.7% |
| Manual recursive implementation | 5.5% |
| mlforecast recursive implementation | 6.0% |

All three land in a similar range. The leakage identified didn't meaningfully inflate the original result, largely because `lag_28`, the model's most important feature by a wide margin, was never affected by it.

The first mlforecast attempt came in at 6.9%, noticeably worse. That turned out to be a feature mismatch, not an implementation quality issue, `is_weekend` and `week_of_year` were missing from that setup. Adding them closed most of the gap. The remaining ~0.5 point difference between the manual and mlforecast versions is reasonable implementation variance, not something worth chasing further.

Thanks to Jan Rathfelder for the original catch, this notebook exists because of that feedback.